<a href="https://colab.research.google.com/github/AureliaVDB/TickIt_Data_Pipeline/blob/main/TickIt_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install pyspark -q

In [18]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType, TimestampType
import os, glob, shutil
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os

In [3]:
spark = (SparkSession.builder
         .appName("TickIt_Medallion_Pipeline")
         .getOrCreate()
         )

In [6]:
from google.colab import drive

drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/TickIt_Project"

RAW = f"{BASE}/raw"  # csv files
BRONZE = f"{BASE}/bronze"  # storage for bronze layer
SILVER = f"{BASE}/silver"  # storage for silver layer
GOLD = f"{BASE}/gold"  # storage for gold layer

for path in [RAW, BRONZE, SILVER, GOLD]:
    os.makedirs(path, exist_ok=True)

Mounted at /content/drive


Bronze Layer

In [7]:
# list of all the source files
raw_files = []
for f in os.listdir(RAW):
    if f.endswith(".csv"):
        raw_files.append(f)

print(f"files in list: {raw_files}")

files in list: ['orders.csv', 'payments.csv', 'events.csv', 'event_categories.csv', 'ticket_types.csv', 'customers.csv', 'reviews.csv']


In [12]:
for file in raw_files:
  input_path = os.path.join(RAW, file)
  table_name = os.path.splitext(file)[0]
  output_path = os.path.join(BRONZE, table_name)

  df = spark.read.csv(input_path, header=True) # load the raw data
  df.write.mode("overwrite").parquet(output_path) # write to bronze layer

  print(f"{table_name.upper()}")
  print()
  print("Schema")
  df.printSchema()
  print(f"Row Count: {df.count()}")
  print()
  print("Sample Records")
  df.show(5)
  print()

ORDERS

Schema
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- ticket_type_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- event_date: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- promo_code: string (nullable = true)

Row Count: 52520

Sample Records
+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+
|      order_id|     customer_id|event_id|ticket_type_id|         order_date|event_date|quantity|order_status|promo_code|
+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+
|    ORD0039854|b2744cf668394fac| EVT0050|       TT00106|2022-08-15 10:00:00|2022-09-03|       2|   completed|      NULL|
|    ORD0002676|ad9181b36ee84121| EVT0092|       TT00194|2021-04-15 23:00:00|20

Silver Layer

In [15]:
bronze_tables = {}

for folder_name in os.listdir(BRONZE):
  folder_path = os.path.join(BRONZE, folder_name)
  if os.path.isdir(folder_path):
    bronze_tables[folder_name] = spark.read.parquet(folder_path)


In [34]:
#Orders

orders_raw = bronze_tables["orders"]

orders_raw.printSchema()

orders_raw.show(5)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- ticket_type_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- event_date: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- promo_code: string (nullable = true)

+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+
|      order_id|     customer_id|event_id|ticket_type_id|         order_date|event_date|quantity|order_status|promo_code|
+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+
|    ORD0039854|b2744cf668394fac| EVT0050|       TT00106|2022-08-15 10:00:00|2022-09-03|       2|   completed|      NULL|
|    ORD0002676|ad9181b36ee84121| EVT0092|       TT00194|2021-04-15 23:00:00|2021-06-30|       6|   completed|      NULL|
|    

In [35]:
# fixing variable type
orders_cleaned = (
    orders_raw.withColumn("quantity", F.col("quantity").cast(IntegerType()))
    .withColumn("order_date", F.to_timestamp(F.col("order_date")))
    .withColumn("event_date", F.to_date(F.col("event_date")))
)

orders_cleaned.printSchema()
orders_cleaned.show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- ticket_type_id: string (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- event_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- promo_code: string (nullable = true)

+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+
|order_id      |customer_id     |event_id|ticket_type_id|order_date         |event_date|quantity|order_status|promo_code|
+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+
|ORD0039854    |b2744cf668394fac|EVT0050 |TT00106       |2022-08-15 10:00:00|2022-09-03|2       |completed   |NULL      |
|ORD0002676    |ad9181b36ee84121|EVT0092 |TT00194       |2021-04-15 23:00:00|2021-06-30|6       |completed   |NULL      |
|OR

In [36]:
# nullable

orders_cleaned.select(
    [
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in [
            "order_id",
            "customer_id",
            "event_id",
            "ticket_type_id",
            "order_date",
            "quantity",
            "promo_code",
        ]
    ]
).show()

+--------+-----------+--------+--------------+----------+--------+----------+
|order_id|customer_id|event_id|ticket_type_id|order_date|quantity|promo_code|
+--------+-----------+--------+--------------+----------+--------+----------+
|       0|        270|       0|             0|         0|       0|     29352|
+--------+-----------+--------+--------------+----------+--------+----------+



In [37]:

missing_customers = orders_cleaned.filter(F.col("customer_id").isNull())

print(f"Total rows with missing customer_id: {missing_customers.count()}")
missing_customers.show(10, truncate=False)


Total rows with missing customer_id: 270
+----------+-----------+--------+--------------+-------------------+----------+--------+------------+----------+
|order_id  |customer_id|event_id|ticket_type_id|order_date         |event_date|quantity|order_status|promo_code|
+----------+-----------+--------+--------------+-------------------+----------+--------+------------+----------+
|ORD0037574|NULL       |EVT0181 |TT00366       |2022-04-12 06:00:00|2022-04-20|4       |completed   |NULL      |
|ORD0003884|NULL       |EVT0340 |TT00705       |2022-10-17 19:00:00|2022-11-13|4       |completed   |STUDENT15 |
|ORD0037061|NULL       |EVT0408 |TT00835       |2025-04-18 08:00:00|2025-05-08|1       |completed   |NULL      |
|ORD0008603|NULL       |EVT0256 |TT00532       |2022-10-30 04:00:00|2022-12-06|2       |completed   |NULL      |
|ORD0012328|NULL       |EVT0118 |TT00240       |2022-06-01 17:00:00|2022-06-16|1       |completed   |NULL      |
|ORD0043240|NULL       |EVT0240 |TT00492       |2024-12

In [38]:
total_orders = orders_cleaned.count()
distinct_orders = orders_cleaned.select("order_id").distinct().count()

print(f"Total Rows: {total_orders}")
print(f"Distinct Order IDs: {distinct_orders}")
print(f"Duplicate Order IDs found: {total_orders - distinct_orders}")

Total Rows: 52520
Distinct Order IDs: 52520
Duplicate Order IDs found: 0


In [39]:
orders_silver = orders_cleaned.fillna({"customer_id": "GUEST"})

orders_silver.select(
    [
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in ["order_id", "customer_id", "quantity", "promo_code"]
    ]
).show()

+--------+-----------+--------+----------+
|order_id|customer_id|quantity|promo_code|
+--------+-----------+--------+----------+
|       0|          0|       0|     29352|
+--------+-----------+--------+----------+



In [40]:
# check quantity for unrealistic number

orders_silver.select("quantity").describe().show()

quantiles = orders_silver.approxQuantile(
    "quantity", [0.25, 0.50, 0.75, 0.95, 0.99, 1.0], 0.01
)
print("Percentiles [25%, 50%, 75%, 95%, 99%, Max]:", quantiles)

+-------+------------------+
|summary|          quantity|
+-------+------------------+
|  count|             52520|
|   mean| 2.547981721249048|
| stddev|1.3551327100983674|
|    min|                 0|
|    max|                 6|
+-------+------------------+

Percentiles [25%, 50%, 75%, 95%, 99%, Max]: [2.0, 2.0, 3.0, 5.0, 6.0, 6.0]


In [41]:
dup_orders = orders_silver.filter(F.col("order_id").contains("_DUP"))

print(f"Total records with '_DUP' in order_id: {dup_orders.count()}")
dup_orders.show(10, truncate=False)

Total records with '_DUP' in order_id: 520
+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+
|order_id      |customer_id     |event_id|ticket_type_id|order_date         |event_date|quantity|order_status|promo_code|
+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+
|ORD0048203_DUP|1e2a435ad7c74203|EVT0160 |TT00325       |2022-03-20 09:00:00|2022-04-02|1       |completed   |NULL      |
|ORD0027374_DUP|e771cb9cbded4054|EVT0076 |TT00155       |2024-03-04 22:00:00|2024-05-31|1       |completed   |NULL      |
|ORD0003043_DUP|bfc8ed144bd74aa8|EVT0240 |TT00492       |2024-07-20 23:00:00|2024-08-17|2       |completed   |SUMMER10  |
|ORD0022897_DUP|c844312a9119439f|EVT0067 |TT00135       |2024-10-03 16:00:00|2024-10-08|1       |completed   |NULL      |
|ORD0021446_DUP|25d94a5b1da34368|EVT0301 |TT00625       |2023-05-29 04:00:00|2023-06-05|6       |comple

In [42]:
orders_silver.filter(
    (F.col("order_id") == "ORD0048203") | (F.col("order_id") == "ORD0048203_DUP")
).show(truncate=False)

+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+
|order_id      |customer_id     |event_id|ticket_type_id|order_date         |event_date|quantity|order_status|promo_code|
+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+
|ORD0048203_DUP|1e2a435ad7c74203|EVT0160 |TT00325       |2022-03-20 09:00:00|2022-04-02|1       |completed   |NULL      |
|ORD0048203    |1e2a435ad7c74203|EVT0160 |TT00325       |2022-03-20 09:00:00|2022-04-02|1       |completed   |NULL      |
+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+



In [33]:
orders_silver = orders_silver.withColumn(
    "order_id", F.regexp_replace(F.col("order_id"), "_DUP", "")
)

orders_silver = orders_silver.dropDuplicates(["order_id"])


In [43]:
orders_silver.groupBy("order_status").count().orderBy(
    F.col("count").desc()
).show(truncate=False)

+------------+-----+
|order_status|count|
+------------+-----+
|completed   |38623|
|cancelled   |8124 |
|confirmed   |3198 |
|no_show     |2575 |
+------------+-----+



In [44]:
orders_silver_path = os.path.join(SILVER, "orders")
orders_silver.write.mode("overwrite").parquet(orders_silver_path)

In [45]:
# Customers
customers_raw = bronze_tables["customers"]

customers_raw.printSchema()

customers_raw.show(5, truncate=False)

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- date_of_birth: string (nullable = true)

+----------------+--------------------------------+----------+---------+-------------------------+-------+------------+--------------+-----------------+-------------+
|customer_id     |customer_unique_id              |first_name|last_name|email                    |city   |country_code|country       |registration_date|date_of_birth|
+----------------+--------------------------------+----------+---------+-------------------------+-------+------------+--------------+-----------------+-------------+
|7ff8bab402524574|4988c52bf1bf47f2b6e3ef3fe0eeb9d3|Simon     |Torre

In [46]:
customers_cleaned = (
    customers_raw
    .withColumn("registration_date", F.to_date(F.col("registration_date")))
    .withColumn("date_of_birth", F.to_date(F.col("date_of_birth")))
)

customers_cleaned.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- date_of_birth: date (nullable = true)



In [47]:
customers_cleaned.select(
    F.count("*").alias("total_rows"),
    F.count_distinct("customer_id").alias("distinct_customer_ids"),
    F.count_distinct("customer_unique_id").alias("distinct_unique_ids"),
    F.sum(F.when(F.col("customer_id").isNull(), 1).otherwise(0)).alias("null_customer_ids")
).show()

+----------+---------------------+-------------------+-----------------+
|total_rows|distinct_customer_ids|distinct_unique_ids|null_customer_ids|
+----------+---------------------+-------------------+-----------------+
|     10800|                10800|              10692|                0|
+----------+---------------------+-------------------+-----------------+



In [48]:
# Trim whitespace from text fields and lowercase email for consistency
customers_silver = (
    customers_cleaned
    .withColumn("first_name", F.trim(F.col("first_name")))
    .withColumn("last_name", F.trim(F.col("last_name")))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("country", F.trim(F.col("country")))
    .withColumn("country_code", F.upper(F.trim(F.col("country_code"))))
)


In [49]:
customers_silver_path = os.path.join(SILVER, "customers")
customers_silver.write.mode("overwrite").parquet(customers_silver_path)

{"ts": "2026-07-31 05:30:02.305", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '22/07/2022' of the type \"STRING\" cannot be cast to \"DATE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "to_date", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o1072.parquet.\n: org.apache.spark.SparkDateTimeException: [CAST_INVALID_INPUT] The value '22/07/2022' of the type \"STRING\" cannot be cast to \"DATE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"to_date\" was called from\njav

DateTimeException: [CAST_INVALID_INPUT] The value '22/07/2022' of the type "STRING" cannot be cast to "DATE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"to_date" was called from
java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
